In [23]:
# ============================================================
# STATISTICAL HYPOTHESIS TESTING USING LOGISTIC REGRESSION
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# ------------------------------------------------------------
# 1. Load the Telco Customer Churn dataset
# ------------------------------------------------------------

churn_df = pd.read_csv('Telco_customer_churn.csv')

# ------------------------------------------------------------
# 2. Define dependent variable
# Churn Value: 1 = Churn, 0 = No Churn
# ------------------------------------------------------------

y = churn_df['Churn Value']

# ------------------------------------------------------------
# 3. Convert numerical variables
# ------------------------------------------------------------

churn_df['Tenure Months'] = pd.to_numeric(
    churn_df['Tenure Months'], errors='coerce'
)

churn_df['Monthly Charges'] = pd.to_numeric(
    churn_df['Monthly Charges'], errors='coerce'
)

churn_df['Total Charges'] = pd.to_numeric(
    churn_df['Total Charges'], errors='coerce'
)

# ------------------------------------------------------------
# 4. Select variables relevant to the hypothesis
# ------------------------------------------------------------

hypothesis_features = [
    'Tenure Months',
    'Total Charges',
    'Monthly Charges',
    'Contract',
    'Internet Service'
]

X = churn_df[hypothesis_features].copy()

# ------------------------------------------------------------
# 5. Handle missing numerical values
# ------------------------------------------------------------

X['Tenure Months'] = X['Tenure Months'].fillna(
    X['Tenure Months'].median()
)

X['Monthly Charges'] = X['Monthly Charges'].fillna(
    X['Monthly Charges'].median()
)

X['Total Charges'] = X['Total Charges'].fillna(
    X['Total Charges'].median()
)

# ------------------------------------------------------------
# 6. Convert categorical variables into dummy variables
# ------------------------------------------------------------

X = pd.get_dummies(
    X,
    columns=['Contract', 'Internet Service'],
    drop_first=True
)

# ------------------------------------------------------------
# 7. Convert Boolean columns to integers
# ------------------------------------------------------------

X = X.astype(float)

# ------------------------------------------------------------
# 8. Add intercept
# ------------------------------------------------------------

X = sm.add_constant(X)

# ------------------------------------------------------------
# 9. Fit Binary Logistic Regression
# ------------------------------------------------------------

logit_model = sm.Logit(y, X)
result = logit_model.fit()

# ------------------------------------------------------------
# 10. Display complete statistical results
# ------------------------------------------------------------

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.428953
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:            Churn Value   No. Observations:                 7043
Model:                          Logit   Df Residuals:                     7035
Method:                           MLE   Df Model:                            7
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                  0.2586
Time:                        17:07:26   Log-Likelihood:                -3021.1
converged:                       True   LL-Null:                       -4075.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                            0.0212      0.193      0.110      0.913

In [24]:
# ============================================================
# P-VALUE TABLE
# ============================================================

results_table = pd.DataFrame({
    'Feature': result.params.index,
    'Coefficient': result.params.values,
    'P-Value': result.pvalues.values,
    'Odds Ratio': np.exp(result.params.values),
    'CI Lower': np.exp(result.conf_int()[0].values),
    'CI Upper': np.exp(result.conf_int()[1].values)
})

# Remove intercept
results_table = results_table[
    results_table['Feature'] != 'const'
]

# Add significance decision
results_table['Significance'] = results_table['P-Value'].apply(
    lambda p: 'Significant (Reject H0)'
    if p < 0.05
    else 'Not Significant (Fail to Reject H0)'
)

# Round values
results_table['Coefficient'] = results_table['Coefficient'].round(4)
results_table['P-Value'] = results_table['P-Value'].round(6)
results_table['Odds Ratio'] = results_table['Odds Ratio'].round(4)
results_table['CI Lower'] = results_table['CI Lower'].round(4)
results_table['CI Upper'] = results_table['CI Upper'].round(4)

print("\n========== HYPOTHESIS TEST RESULTS ==========\n")
print(results_table.to_string(index=False))


========== HYPOTHESIS TEST RESULTS ==========

                     Feature  Coefficient  P-Value  Odds Ratio  CI Lower  CI Upper                        Significance
               Tenure Months      -0.0579 0.000000      0.9437    0.9324    0.9551             Significant (Reject H0)
               Total Charges       0.0003 0.000005      1.0003    1.0002    1.0005             Significant (Reject H0)
             Monthly Charges      -0.0036 0.290758      0.9964    0.9897    1.0031 Not Significant (Fail to Reject H0)
           Contract_One year      -0.8690 0.000000      0.4194    0.3422    0.5140             Significant (Reject H0)
           Contract_Two year      -1.7816 0.000000      0.1684    0.1203    0.2356             Significant (Reject H0)
Internet Service_Fiber optic       1.1059 0.000000      3.0221    2.3510    3.8847             Significant (Reject H0)
         Internet Service_No      -1.0925 0.000000      0.3354    0.2473    0.4548             Significant (Reject H0)


In [25]:
# ============================================================
# TENURE-SPECIFIC HYPOTHESIS RESULT
# ============================================================

tenure_coef = result.params['Tenure Months']
tenure_p = result.pvalues['Tenure Months']
tenure_odds = np.exp(tenure_coef)

print("\n========== TENURE HYPOTHESIS TEST ==========")

print(f"Tenure coefficient : {tenure_coef:.4f}")
print(f"Tenure p-value     : {tenure_p}")
print(f"Tenure odds ratio  : {tenure_odds:.4f}")

if tenure_p < 0.05:
    print("Result: Tenure Months has a statistically significant association with churn.")
    
    if tenure_coef < 0:
        print("Direction: Higher tenure is associated with lower odds of churn.")
        print("This supports the hypothesis that shorter-tenure customers have higher churn risk.")
    else:
        print("Direction: Higher tenure is associated with higher odds of churn.")
else:
    print("Result: Tenure Months is not statistically significant at the 5% level.")


========== TENURE HYPOTHESIS TEST ==========
Tenure coefficient : -0.0579
Tenure p-value     : 2.968253278786819e-21
Tenure odds ratio  : 0.9437
Result: Tenure Months has a statistically significant association with churn.
Direction: Higher tenure is associated with lower odds of churn.
This supports the hypothesis that shorter-tenure customers have higher churn risk.


In [27]:
# ============================================================
# OVERALL LIKELIHOOD RATIO TEST
# ============================================================

# Null model: intercept only
X_null = pd.DataFrame({
    'const': np.ones(len(y))
})

null_model = sm.Logit(y, X_null).fit(disp=False)

# Full model log-likelihood
full_log_likelihood = result.llf

# Null model log-likelihood
null_log_likelihood = null_model.llf

# Likelihood Ratio statistic
lr_statistic = 2 * (full_log_likelihood - null_log_likelihood)

# Difference in degrees of freedom
df = result.df_model

# Overall p-value
overall_p_value = stats.chi2.sf(lr_statistic, df)

print("\n========== OVERALL HYPOTHESIS TEST ==========")

print(f"Likelihood Ratio Statistic : {lr_statistic}")
print(f"Degrees of Freedom         : {int(df)}")
print(f"Overall p-value            : {overall_p_value}")

if overall_p_value < 0.05:
    print("\nDecision: Reject H0")
    print("Conclusion: The predictor variables jointly provide")
    print("statistically significant information about customer churn.")
else:
    print("\nDecision: Fail to Reject H0")
    print("Conclusion: The predictor variables do not jointly")
    print("provide statistically significant information about customer churn.")


========== OVERALL HYPOTHESIS TEST ==========
Likelihood Ratio Statistic : 2107.9137428878075
Degrees of Freedom         : 7
Overall p-value            : 0.0

Decision: Reject H0
Conclusion: The predictor variables jointly provide
statistically significant information about customer churn.
